# TD 4 — Corrigé

**Analyse des données — L3 Économie**

Mis en ligne après la séance. Page du fichier : <https://www.insee.fr/fr/statistiques/8680726>

## Partie 1 — Charger

In [ ]:
import pandas as pd
import numpy as np

URL = "https://www.insee.fr/fr/statistiques/fichier/8680726/ensemble.xlsx"

com = pd.read_excel(URL, sheet_name="Communes",     header=7)
dep = pd.read_excel(URL, sheet_name="Départements", header=7)

print(com.shape, dep.shape)

**Réponse 1.** Les feuilles commencent par sept lignes de titre, de source et de notes. Sans `header=7`, pandas prendrait le titre du tableau pour des noms de colonnes. Il faut **ouvrir le fichier d'abord** et compter : c'est le réflexe annoncé en séance 2.

**Réponse 2.** **34 904 lignes** d'un côté, **100** de l'autre.

- Dans `Communes`, une ligne décrit **une commune** — ou, on le verra, un arrondissement municipal.
- Dans `Départements`, une ligne décrit **un département**.

Les deux tableaux n'ont pas la même unité statistique : c'est précisément pour cela qu'on les fusionne.

## Partie 2 — La clé

In [ ]:
print(com.dtypes)
com[["Code département", "Code commune", "Nom de la commune"]].head(3)

**Réponse 3.** `Code département` est du **texte** : `"01"`, `"2A"`, `"971"`. Le zéro initial est préservé, et la Corse s'écrit en lettres.

`Code commune` est un **entier** : L'Abergement-Clémenciat porte le code **`1`**, et non `001`. Les zéros de tête ont disparu.

C'est le même piège que le code commune de la séance 2, sous une autre forme : un identifiant stocké comme un nombre.

In [ ]:
com["CODGEO"] = com["Code département"] + com["Code commune"].astype(str).str.zfill(3)

com.loc[com["Nom de la commune"] == "L' Abergement-Clémenciat", "CODGEO"]

**Réponse 4.** `01001`. `.str.zfill(3)` complète par des zéros à gauche jusqu'à trois caractères : `1` devient `001`.

Sans `zfill`, on obtiendrait `011` — un code qui existe peut-être ailleurs, et qui désignerait donc la **mauvaise commune**. Aucune erreur ne le signalerait.

In [ ]:
com["CODGEO"].str.len().value_counts()

**Réponse 5.** **34 788** codes font cinq caractères, **116** en font six.

Les exceptions sont les communes d'**outre-mer**, dont le code département compte trois chiffres (`971`, `972`, …). La règle « département + trois chiffres » y produit six caractères, alors que le code officiel en compte cinq.

Ces 116 communes se décomposent en 112 communes des départements d'outre-mer et 4 communes de collectivités d'outre-mer, rencontrées à la partie 3.

> Une règle qui marche sur 99,7 % des lignes n'est pas une règle juste. **Vérifier la longueur** d'un identifiant construit est un contrôle d'une ligne, et il révèle ce que l'œil ne voit pas.

## Partie 3 — Fusionner

In [ ]:
f = com.merge(
    dep[["Code département", "Nom du département", "Population municipale"]],
    on="Code département",
    how="left",
    indicator=True,
    suffixes=("_com", "_dep"))

print("lignes avant :", len(com), "| apres :", len(f))

**Réponse 6.** **34 904** avant, **34 904** après.

Le nombre ne change pas parce que la relation est **plusieurs-à-un** : chaque commune a au plus un département, et chaque département n'apparaît qu'une fois dans sa feuille. Chaque ligne de gauche reçoit donc au plus un partenaire.

Si le nombre de lignes **augmentait**, ce serait le signe qu'un code département figure plusieurs fois à droite : chaque commune serait dupliquée. C'est le cas le plus dangereux, parce qu'il ne produit aucune erreur — voir la question 12.

In [ ]:
print(f["_merge"].value_counts())
f.loc[f["_merge"] == "left_only", ["Code département", "Code commune", "Nom de la commune"]]

**Réponse 7.** **34 900** communes appariées, **4** sans département :

| Code département | Commune |
|---|---|
| 975 | Miquelon-Langlade |
| 975 | Saint-Pierre |
| 977 | Saint-Barthélemy |
| 978 | Saint-Martin |

**Réponse 8.** Ces codes désignent des **collectivités d'outre-mer** — Saint-Pierre-et-Miquelon, Saint-Barthélemy, Saint-Martin. Elles ont un statut propre, distinct de celui de département : elles figurent donc parmi les communes, mais pas dans la feuille des départements.

Ce n'est **pas une erreur** de fusion. C'est une information sur l'organisation administrative, que seule la colonne `_merge` a rendue visible.

> Une fusion avec `how="inner"` aurait supprimé ces quatre communes sans le moindre signal. D'où `how="left"` et `indicator=True` : on garde tout, et on regarde ce qui ne s'apparie pas.

## Partie 4 — Interpréter

In [ ]:
f["part"] = 100 * f["Population municipale_com"] / f["Population municipale_dep"]

f.sort_values("part", ascending=False)[["Nom de la commune", "Nom du département", "part"]].head(5)

**Réponse 9.**

| Commune | Département | Part |
|---|---|---|
| Ajaccio | Corse-du-Sud | 45,3 % |
| Toulouse | Haute-Garonne | 35,0 % |
| Limoges | Haute-Vienne | 34,8 % |
| Belfort | Territoire de Belfort | 32,7 % |
| Nice | Alpes-Maritimes | 31,7 % |

**Ajaccio** pèse près de la moitié de son département. Ce n'est pas qu'Ajaccio soit une grande ville : c'est que la Corse-du-Sud est **peu peuplée**. Une part mesure un rapport, pas une taille — et le dénominateur compte autant que le numérateur.

In [ ]:
f[f["Nom de la commune"].str.contains("Marseille")][["Nom de la commune", "part"]]

**Réponse 10.** Dans ce fichier, **Marseille n'est pas une commune** : elle apparaît découpée en **arrondissements municipaux**, chacun pesant de 1 à 3 % des Bouches-du-Rhône. Il en va de même pour Paris et Lyon, qui relèvent du même statut.

Aucun arrondissement ne peut donc arriver en tête. Pour obtenir la part de Marseille entière, il faudrait **agréger** ses arrondissements avant de calculer la part.

> Avant d'interpréter un classement, il faut savoir ce que chaque ligne représente. La question 2 n'était pas une formalité.

## Pour ceux qui ont terminé

**Réponse 11.** La recherche ramène aussi **Marseillette**, une commune de l'Aude. De même, chercher « Lyon » ramène *Cognat-Lyonne*, dans l'Allier.

`str.contains` cherche une **sous-chaîne**, pas un nom exact. Rechercher par nom est fragile — homonymes, accents, articles. **C'est pour cela qu'on fusionne sur des codes et non sur des noms.**

In [ ]:
f2 = com.merge(dep[["Code département", "Nom du département"]],
               on="Code département", how="left", validate="m:1")
print(len(f2))

**Réponse 12.** `validate="m:1"` vérifie que la clé est **unique à droite** : plusieurs communes peuvent pointer vers un même département, mais chaque département ne doit apparaître qu'une fois.

Si la feuille des départements contenait deux fois le même code, la fusion **s'arrêterait** avec une `MergeError`. Sans `validate`, elle aurait réussi en silence — en dupliquant toutes les communes de ce département.

> `validate` transforme une erreur silencieuse en erreur bruyante. C'est exactement ce qu'on cherche.

In [ ]:
d = com["Code département"]
n = com["Code commune"].astype(str).str.zfill(3)

# Outre-mer : le troisieme chiffre du departement est deja le premier du code commune
com["CODGEO"] = np.where(d.str.len() == 3, d.str[:2] + n, d + n)

print(com["CODGEO"].str.len().value_counts())
com.loc[com["Code département"].isin(["975", "977"]), ["Nom de la commune", "CODGEO"]]

**Réponse 13.** Pour les codes département à trois chiffres, on ne garde que les deux premiers, puisque le troisième est déjà le premier chiffre du code commune : Miquelon-Langlade, `975` et `501`, donne `97501` — son code officiel.

La dernière sortie ne doit plus afficher que des longueurs de **5**. C'est la vérification qui valide la correction : on ne suppose pas qu'elle marche, on le constate.

## Erreurs fréquentes

| Erreur | Symptôme | Ce qu'elle enseigne |
|---|---|---|
| Oublier `header=7` | colonnes nommées `Unnamed` | Ouvrir un fichier avant de le charger |
| Concaténer sans `zfill` | `011` au lieu de `01001` | Un identifiant n'est jamais un nombre |
| Ne pas vérifier la longueur | 116 codes faux, invisibles | Une règle se vérifie, elle ne se suppose pas |
| `how="inner"` | 4 communes disparues sans signal | Garder tout, puis regarder ce qui manque |
| Oublier `indicator=True` | impossible de savoir ce qui ne s'est pas apparié | La colonne `_merge` est le diagnostic |
| Interpréter le classement tel quel | « Marseille est absente » | Savoir ce que représente une ligne |